# 03 — Weather Data Quality Analysis

**RiskWeave | ERA5 daily weather | Sri Lanka**

Coverage: 23 December 2006–19 December 2025. Expected coverage: 25 districts × 6,937 days = 173,425 district-date rows.

This notebook analyses the consolidated master CSV without overwriting it, deleting outliers, or imputing values. Reports are saved separately under `outputs/weather_quality/`.

ERA5 is gridded reanalysis data, not direct measurements from district weather stations. The requested representative coordinates and API-returned ERA5 grid coordinates are distinct.


## 1. Import libraries

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")


## 2. Configure paths and expected coverage

In [ ]:
CURRENT_DIR = Path.cwd()
PROJECT_DIR = CURRENT_DIR.parent if CURRENT_DIR.name.lower() == "notebooks" else CURRENT_DIR

MASTER_FILE = (
    PROJECT_DIR / "data" / "raw" / "weather_era5_2006_2025"
    / "weather_daily_era5_2006-12-23_to_2025-12-19.csv"
)
OUTPUT_DIR = PROJECT_DIR / "outputs" / "weather_quality"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

START_DATE = pd.Timestamp("2006-12-23")
END_DATE = pd.Timestamp("2025-12-19")
EXPECTED_DAYS_PER_DISTRICT = len(pd.date_range(START_DATE, END_DATE, freq="D"))
EXPECTED_DISTRICTS = 25
EXPECTED_TOTAL_ROWS = EXPECTED_DAYS_PER_DISTRICT * EXPECTED_DISTRICTS

WEATHER_VARIABLES = [
    "precipitation_sum", "rain_sum", "precipitation_hours",
    "temperature_2m_mean", "temperature_2m_max", "temperature_2m_min",
    "relative_humidity_2m_mean", "relative_humidity_2m_max",
    "relative_humidity_2m_min", "dew_point_2m_mean",
    "wind_speed_10m_mean", "wind_speed_10m_max",
    "wind_direction_10m_dominant", "cloud_cover_mean",
    "shortwave_radiation_sum", "et0_fao_evapotranspiration"
]

print("Project:", PROJECT_DIR)
print("Master CSV:", MASTER_FILE)
print("File exists:", MASTER_FILE.exists())
print("Expected days/district:", EXPECTED_DAYS_PER_DISTRICT)
print("Expected total rows:", EXPECTED_TOTAL_ROWS)


## 3. Load master weather dataset

In [ ]:
if not MASTER_FILE.exists():
    raise FileNotFoundError(f"Master weather CSV not found: {MASTER_FILE}")

weather = pd.read_csv(MASTER_FILE)
print("Shape:", weather.shape)
display(weather.head())


## 4. Check required columns and data types

In [ ]:
required = ["district", "date"] + WEATHER_VARIABLES
missing_columns = [c for c in required if c not in weather.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

weather["district"] = weather["district"].astype("string").str.strip()
weather["date"] = pd.to_datetime(weather["date"], errors="coerce")
for col in WEATHER_VARIABLES:
    weather[col] = pd.to_numeric(weather[col], errors="coerce")

display(weather.dtypes.astype(str).to_frame("data_type"))
print("Invalid dates:", weather["date"].isna().sum())


## 5. Overall structure and completeness

In [ ]:
overall = pd.DataFrame({
    "metric": [
        "Rows", "Districts", "Expected districts", "Weather variables",
        "First date", "Last date", "Duplicate district-date rows",
        "Invalid dates", "Missing district labels", "Expected total rows"
    ],
    "value": [
        len(weather), weather["district"].nunique(), EXPECTED_DISTRICTS,
        len(WEATHER_VARIABLES), weather["date"].min(), weather["date"].max(),
        weather.duplicated(["district", "date"]).sum(),
        weather["date"].isna().sum(), weather["district"].isna().sum(),
        EXPECTED_TOTAL_ROWS
    ]
})
display(overall)


## 6. Missing values

In [ ]:
missing_summary = weather[WEATHER_VARIABLES].isna().sum().to_frame("missing_count")
missing_summary["missing_percentage"] = missing_summary["missing_count"] / len(weather) * 100
display(missing_summary)
missing_summary.to_csv(OUTPUT_DIR / "weather_missing_values_summary.csv")


## 7. Duplicate district-date records

In [ ]:
duplicate_mask = weather.duplicated(["district", "date"], keep=False)
duplicates = weather.loc[duplicate_mask].sort_values(["district", "date"])
print("Duplicate rows:", len(duplicates))
display(duplicates.head(30))
duplicates.to_csv(OUTPUT_DIR / "weather_duplicate_district_date_records.csv", index=False)


## 8. Date continuity and row counts by district

In [ ]:
expected_dates = pd.date_range(START_DATE, END_DATE, freq="D")
continuity_rows = []

for district in sorted(weather["district"].dropna().unique()):
    part = weather.loc[weather["district"] == district]
    dates = pd.DatetimeIndex(part["date"].dropna().unique()).sort_values()
    missing_dates = expected_dates.difference(dates)
    unexpected_dates = dates.difference(expected_dates)
    continuity_rows.append({
        "district": district,
        "rows": len(part),
        "unique_dates": part["date"].nunique(),
        "missing_dates": len(missing_dates),
        "unexpected_dates": len(unexpected_dates),
        "duplicate_pairs": part.duplicated(["district", "date"]).sum(),
        "first_date": part["date"].min(),
        "last_date": part["date"].max(),
        "row_count_ok": len(part) == EXPECTED_DAYS_PER_DISTRICT,
        "date_coverage_ok": len(missing_dates) == 0 and len(unexpected_dates) == 0
    })

continuity = pd.DataFrame(continuity_rows)
display(continuity)
continuity.to_csv(OUTPUT_DIR / "weather_date_continuity_by_district.csv", index=False)


## 9. Descriptive statistics

In [ ]:
descriptive = weather[WEATHER_VARIABLES].describe().T
descriptive["missing_count"] = weather[WEATHER_VARIABLES].isna().sum()
descriptive["unique_values"] = weather[WEATHER_VARIABLES].nunique()
display(descriptive)
descriptive.to_csv(OUTPUT_DIR / "weather_descriptive_statistics.csv")


## 10. Variable units reference

| Variable | Unit |
|---|---|
| `precipitation_sum`, `rain_sum` | mm |
| `precipitation_hours` | hours |
| `temperature_2m_mean`, `temperature_2m_max`, `temperature_2m_min`, `dew_point_2m_mean` | °C |
| `relative_humidity_2m_mean`, `relative_humidity_2m_max`, `relative_humidity_2m_min`, `cloud_cover_mean` | % |
| `wind_speed_10m_mean`, `wind_speed_10m_max` | km/h |
| `wind_direction_10m_dominant` | degrees |
| `shortwave_radiation_sum` | MJ/m² |
| `et0_fao_evapotranspiration` | mm |

These are the requested Open-Meteo daily variable units. Confirm them against the API response metadata retained in your district JSON files.


## 11. Broad physical-range screening

In [ ]:
# Broad screening bounds only. These are not automatic deletion rules.
# A flagged value should be investigated; a climate extreme may be genuine.
limits = {
    "precipitation_sum": (0, None), "rain_sum": (0, None),
    "precipitation_hours": (0, 24),
    "temperature_2m_mean": (-90, 65), "temperature_2m_max": (-90, 65),
    "temperature_2m_min": (-90, 65), "dew_point_2m_mean": (-100, 65),
    "relative_humidity_2m_mean": (0, 100),
    "relative_humidity_2m_max": (0, 100),
    "relative_humidity_2m_min": (0, 100),
    "wind_speed_10m_mean": (0, None), "wind_speed_10m_max": (0, None),
    "wind_direction_10m_dominant": (0, 360),
    "cloud_cover_mean": (0, 100),
    "shortwave_radiation_sum": (0, None),
    "et0_fao_evapotranspiration": (0, None)
}
range_rows = []
for variable, (lower, upper) in limits.items():
    s = weather[variable]
    below = s < lower if lower is not None else pd.Series(False, index=s.index)
    above = s > upper if upper is not None else pd.Series(False, index=s.index)
    range_rows.append({
        "variable": variable, "lower_limit": lower, "upper_limit": upper,
        "below_lower": int(below.sum()), "above_upper": int(above.sum()),
        "total_flagged": int((below | above).sum())
    })
range_screening = pd.DataFrame(range_rows)
display(range_screening)
range_screening.to_csv(OUTPUT_DIR / "weather_physical_range_screening.csv", index=False)


## 12. Logical consistency checks

In [ ]:
logical_checks = {
    "temperature_min_above_mean": weather["temperature_2m_min"] > weather["temperature_2m_mean"],
    "temperature_mean_above_max": weather["temperature_2m_mean"] > weather["temperature_2m_max"],
    "temperature_min_above_max": weather["temperature_2m_min"] > weather["temperature_2m_max"],
    "humidity_min_above_mean": weather["relative_humidity_2m_min"] > weather["relative_humidity_2m_mean"],
    "humidity_mean_above_max": weather["relative_humidity_2m_mean"] > weather["relative_humidity_2m_max"],
    "humidity_min_above_max": weather["relative_humidity_2m_min"] > weather["relative_humidity_2m_max"],
    "wind_mean_above_max": weather["wind_speed_10m_mean"] > weather["wind_speed_10m_max"],
    "rain_above_precipitation": weather["rain_sum"] > weather["precipitation_sum"]
}
logical_summary = pd.DataFrame([
    {"check": name, "flagged_rows": int(mask.fillna(False).sum())}
    for name, mask in logical_checks.items()
])
display(logical_summary)
logical_summary.to_csv(OUTPUT_DIR / "weather_logical_consistency_checks.csv", index=False)


## 13. Statistical outlier screening (IQR)

In [ ]:
# IQR flags are exploratory only. Do not remove climate extremes automatically.
iqr_rows, flagged_parts = [], []
for variable in WEATHER_VARIABLES:
    q1 = weather[variable].quantile(0.25)
    q3 = weather[variable].quantile(0.75)
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    mask = (weather[variable] < low) | (weather[variable] > high)
    iqr_rows.append({
        "variable": variable, "q1": q1, "q3": q3, "iqr": iqr,
        "lower_bound": low, "upper_bound": high,
        "flagged_count": int(mask.sum()),
        "flagged_percentage": float(mask.mean() * 100)
    })
    if mask.any():
        p = weather.loc[mask, ["district", "date", variable]].copy()
        p = p.rename(columns={variable: "value"})
        p["variable"] = variable
        flagged_parts.append(p)

iqr_summary = pd.DataFrame(iqr_rows)
display(iqr_summary)
iqr_summary.to_csv(OUTPUT_DIR / "weather_iqr_outlier_summary.csv", index=False)

iqr_records = (
    pd.concat(flagged_parts, ignore_index=True)
    if flagged_parts else pd.DataFrame(columns=["district", "date", "value", "variable"])
)
iqr_records.to_csv(OUTPUT_DIR / "weather_iqr_flagged_records.csv", index=False)
print("IQR flagged observations:", len(iqr_records))


## 14. District coordinate and ERA5 grid metadata

In [ ]:
metadata_cols = [
    c for c in [
        "requested_latitude", "requested_longitude",
        "era5_grid_latitude", "era5_grid_longitude",
        "era5_grid_elevation_m", "model", "timezone"
    ] if c in weather.columns
]

if metadata_cols:
    district_metadata = weather.groupby("district")[metadata_cols].first().reset_index()
    metadata_variation = weather.groupby("district")[metadata_cols].nunique(dropna=False)
    display(district_metadata)
    print("Within-district metadata variation:")
    display(metadata_variation)
    district_metadata.to_csv(OUTPUT_DIR / "weather_district_grid_metadata.csv", index=False)
    if {"era5_grid_latitude", "era5_grid_longitude"}.issubset(district_metadata.columns):
        print("Distinct ERA5 grid coordinate pairs:",
              district_metadata[["era5_grid_latitude", "era5_grid_longitude"]].drop_duplicates().shape[0])
else:
    print("Coordinate metadata columns are absent. Check collection output schema.")


## 15. Plot distributions of selected variables

In [ ]:
for variable in [
    "precipitation_sum", "temperature_2m_mean",
    "relative_humidity_2m_mean", "wind_speed_10m_mean"
]:
    plt.figure(figsize=(9, 4))
    weather[variable].dropna().hist(bins=50)
    plt.title(f"Distribution of {variable}")
    plt.xlabel(variable)
    plt.ylabel("Daily district records")
    plt.tight_layout()
    plt.show()


## 16. Plot time series for one district

In [ ]:
selected_district = "Colombo"  # Change to an exact district value if required
if selected_district not in weather["district"].dropna().unique():
    raise ValueError(f"{selected_district} not found in the dataset.")

district_weather = weather.loc[weather["district"] == selected_district].sort_values("date")
for variable in ["precipitation_sum", "temperature_2m_mean", "relative_humidity_2m_mean"]:
    plt.figure(figsize=(14, 4))
    plt.plot(district_weather["date"], district_weather[variable], linewidth=0.6)
    plt.title(f"{selected_district}: daily {variable}")
    plt.xlabel("Date")
    plt.ylabel(variable)
    plt.tight_layout()
    plt.show()


## 17. Compare district date sequences

In [ ]:
date_sequences = {}
for district in sorted(weather["district"].dropna().unique()):
    date_sequences[district] = (
        weather.loc[weather["district"] == district, "date"]
        .dropna().sort_values().reset_index(drop=True)
    )

reference_district = next(iter(date_sequences))
reference_dates = date_sequences[reference_district]
sequence_report = pd.DataFrame([
    {
        "district": district,
        "same_sequence_as_reference": dates.equals(reference_dates),
        "reference_district": reference_district,
        "date_count": len(dates)
    }
    for district, dates in date_sequences.items()
])
display(sequence_report)
sequence_report.to_csv(OUTPUT_DIR / "weather_date_sequence_comparison.csv", index=False)


## 18. Save consolidated quality-control summary

In [ ]:
summary = {
    "master_file": str(MASTER_FILE),
    "rows": int(len(weather)),
    "districts": int(weather["district"].nunique()),
    "weather_variables": int(len(WEATHER_VARIABLES)),
    "expected_start": str(START_DATE.date()),
    "expected_end": str(END_DATE.date()),
    "observed_start": str(weather["date"].min().date()) if weather["date"].notna().any() else None,
    "observed_end": str(weather["date"].max().date()) if weather["date"].notna().any() else None,
    "expected_rows": int(EXPECTED_TOTAL_ROWS),
    "duplicate_district_date_rows": int(weather.duplicated(["district", "date"]).sum()),
    "invalid_dates": int(weather["date"].isna().sum()),
    "missing_weather_values": int(weather[WEATHER_VARIABLES].isna().sum().sum()),
    "districts_with_expected_rows": int((continuity["rows"] == EXPECTED_DAYS_PER_DISTRICT).sum()),
    "districts_with_complete_date_coverage": int(continuity["date_coverage_ok"].sum()),
    "note": "Outlier and range flags require review. No records were automatically deleted or imputed."
}
display(pd.DataFrame([{"metric": k, "value": v} for k, v in summary.items()]))

with open(OUTPUT_DIR / "weather_quality_control_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("Reports saved in:", OUTPUT_DIR)


## 19. Interpretation and next step

Review the reports under `outputs/weather_quality/`.

- IQR outliers and physical-range flags are **review candidates**, not automatic errors. Climate extremes may be real.
- Do not delete or impute values without investigating the district, date, variable and source.
- ERA5 grid-cell estimates are not station observations.
- For dengue modelling, aggregate daily weather over each record's actual `week_start` and `week_end` dates. Do not assume all dengue reporting periods are standard Monday–Sunday weeks.
- For forecasting, only use information that would have been available at the forecast issue time. Future observed weather must not be used as a feature for a past forecast origin.

The original master CSV is not overwritten. The next step after reviewing these outputs is to align daily weather with the dengue reporting intervals.
